# Анализ вариабельности сердечного ритма (ВРС / HRV)Полный пайплайн анализа ВСР в Python — статистические, геометрические, корреляционные, спектральные методы и комплексная оценка ПАРС по Баевскому.| Раздел | Метод | Ключевые показатели ||---|---|---|| 1 | Статистический анализ | HR, SDNN, CV, RMSSD, pNN50 || 2 | Геометрический анализ | Мо, АМо, MxDMn, индекс напряжения SI || 3 | Корреляционный анализ | CC1, CC0, скаттерограмма Пуанкаре || 4 | Спектральный анализ (FFT) | VLF, LF, HF, IC, LF/HF, ISCA || 5 | Комплексная оценка | ПАРС (1–10 баллов) |**Данные:** последовательность RR-интервалов в мс, формат CSV (один столбец, без заголовка). По умолчанию загружается синтетический пример из `data/rr_intervals.csv` — заменить на свои данные.

## 0. Импорты и загрузка данных

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom scipy.interpolate import CubicSplinefrom pathlib import Pathplt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})DATA_PATH = Path("data/rr_intervals.csv")FIGURES_DIR = Path("figures")FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
# RR-интервалы: одна колонка в мс, строки с # пропускаютсяdf = pd.read_csv(DATA_PATH, header=None, names=["RR"], comment="#")rr = df["RR"].to_numpy(dtype=float)print(f"Загружено интервалов : {len(rr)}")print(f"Диапазон             : {rr.min():.0f} – {rr.max():.0f} мс")print(f"Среднее              : {rr.mean():.1f} мс")print(f"Длительность записи  : {rr.sum()/1000:.1f} с  ({rr.sum()/60000:.1f} мин)")

---## 1. Статистические и геометрические показатели### Временные показатели- **HR (ЧСС)** — среднее число ударов в минуту.- **SDNN** — стандартное отклонение всех RR. Суммарный показатель ВСР. Норма: 40–80 мс.- **CV** — коэффициент вариации (SDNN / M × 100%).- **RMSSD** — среднеквадратичная разность соседних интервалов. Маркёр парасимпатики. Норма: 20–50 мс.- **pNN50** — доля пар соседних интервалов с разностью > 50 мс. Норма: > 20%.

In [ ]:
n = len(rr)M  = rr.mean()HR = n * 60_000 / rr.sum()SDNN  = np.sqrt(((rr - M)**2).sum() / (n - 1))CV    = SDNN / M * 100diff  = rr[1:] - rr[:-1]RMSSD = np.sqrt((diff**2).mean())pNN50 = (np.abs(diff) > 50).sum() / len(diff) * 100print(f"M      = {M:.1f} мс")print(f"HR     = {HR:.1f} уд/мин   [норма: 60–80]")print(f"SDNN   = {SDNN:.2f} мс       [норма: 40–80 мс]")print(f"CV     = {CV:.2f} %")print(f"RMSSD  = {RMSSD:.2f} мс       [норма: 20–50 мс]")print(f"pNN50  = {pNN50:.2f} %         [норма: > 20 %]")

### Геометрические показатели (вариационная пульсометрия)- **Мо (мода)** — наиболее частое значение RR.- **АМо** — доля кардиоинтервалов в модальном классе.- **MxDMn** — вариационный размах (max − min). Норма: 150–300 мс.- **ИН (SI)** — индекс напряжения Баевского: `АМо / (2 × Мо[с] × MxDMn[с])`. Норма: 80–150 у.е.

In [ ]:
dx = 50  # ширина бина, мсbins = np.arange(400, 1351, dx)counts, edges = np.histogram(rr, bins=bins)centers = (edges[:-1] + edges[1:]) / 2density = counts / (n * dx)mo_idx = counts.argmax()Mo     = centers[mo_idx]AMo    = counts[mo_idx] / n * 100MxDMn  = rr.max() - rr.min()SI     = AMo / (2 * (Mo / 1000) * (MxDMn / 1000))print(f"Мо     = {Mo:.0f} мс")print(f"АМо    = {AMo:.1f} %       [норма: 30–50 %]")print(f"MxDMn  = {MxDMn:.0f} мс      [норма: 150–300 мс]")print(f"ИН     = {SI:.1f} у.е.    [норма: 80–150]")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))ax.bar(centers, density, width=48, color="steelblue",       edgecolor="white", linewidth=0.6, label="Плотность вероятности")ax.axvline(Mo, color="crimson", linewidth=2, linestyle="--",           label=f"Мо = {Mo:.0f} мс  (АМо = {AMo:.1f}%)")ax.axvline(M, color="orange", linewidth=1.5, linestyle=":",           label=f"M = {M:.1f} мс")ax.set_xlabel("RR-интервал, мс"); ax.set_ylabel("Плотность вероятности, 1/мс")ax.set_title("Вариационная пульсограмма", fontsize=14, fontweight="bold")ax.set_xticks(edges); ax.set_xticklabels(edges.astype(int), rotation=45, fontsize=8)ax.legend(); ax.grid(axis="y", alpha=0.3)plt.tight_layout()fig.savefig(FIGURES_DIR / "fig1_histogram.png")plt.show()

---## 2. Корреляционный анализ: АКФ и скаттерограмма### Автокорреляционная функция (АКФ)АКФ показывает, насколько ряд RR коррелирует сам с собой при сдвиге на k ударов.- **CC1 = r(k=1)** — корреляция при первом сдвиге. Мера инерционности ритма.- **CC0** — первый сдвиг, где r < 0. Связан с периодом доминирующей волны: T ≈ 2 × CC0 × M.

In [ ]:
def autocorr_pearson(x, k):    a, b = x[:len(x)-k], x[k:]    m = len(a)    num = m * np.dot(a, b) - a.sum() * b.sum()    da  = np.sqrt(m * np.dot(a, a) - a.sum()**2)    db  = np.sqrt(m * np.dot(b, b) - b.sum()**2)    return float(num / (da * db))max_lag = min(100, n // 3)lags = np.arange(0, max_lag + 1)cc   = np.array([autocorr_pearson(rr, int(k)) for k in lags])CC1 = float(cc[1])neg = np.where(cc < 0)[0]CC0 = int(neg[0]) if len(neg) > 0 else Noneprint(f"CC1 = {CC1:.4f}   (корреляция при k=1)")if CC0:    period_s = 2 * CC0 * M / 1000    print(f"CC0 = {CC0} сдвигов  → период ≈ {period_s:.1f} с  ({1/period_s:.3f} Гц)")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))ax.plot(lags, cc, color="steelblue", linewidth=1.5, marker=".", markersize=4)ax.axhline(0, color="crimson", linewidth=0.9, linestyle="--", label="r = 0")ax.axvline(1, color="green", linewidth=1.2, linestyle=":", label=f"CC1 = {CC1:.3f}")if CC0:    ax.axvline(CC0, color="orange", linewidth=1.2, linestyle=":", label=f"CC0 = {CC0}")ax.set_xlabel("Сдвиг k, удары"); ax.set_ylabel("r(k)")ax.set_title("Автокорреляционная функция кардиоинтервалов", fontsize=14, fontweight="bold")ax.legend(); ax.grid(alpha=0.3)plt.tight_layout()fig.savefig(FIGURES_DIR / "fig2_autocorrelation.png")plt.show()

### Скаттерограмма (диаграмма Пуанкаре)Каждая точка: **(RR[n], RR[n+1])**.Форма облака: эллипс вдоль биссектрисы — норма (дыхательная аритмия).Длина эллипса ≈ MxDMn (коррелирует с HF), ширина — с LF.> Этот метод также используется для **детектирования эктопических сокращений** (экстрасистол) — они выпадают из основного облака как отдельные точки.

In [ ]:
x_sc, y_sc = rr[:-1], rr[1:]lo, hi = rr.min() - 30, rr.max() + 30fig, ax = plt.subplots(figsize=(7, 7))ax.scatter(x_sc, y_sc, alpha=0.45, s=18, color="steelblue", label=f"Пары (n={len(x_sc)})")ax.plot([lo, hi], [lo, hi], color="crimson", linewidth=1.2,        linestyle="--", label="y = x (биссектриса)")ax.set_xlabel("RR[n], мс"); ax.set_ylabel("RR[n+1], мс")ax.set_title("Корреляционная ритмограмма (скаттерограмма)", fontsize=14, fontweight="bold")ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")ax.legend(); ax.grid(alpha=0.3)plt.tight_layout()fig.savefig(FIGURES_DIR / "fig3_scatterogram.png")plt.show()

---## 3. Спектральный анализ ВСР (FFT)**Конвейер обработки:**1. **Кубическая сплайн-интерполяция** → равномерная сетка (fs = 4 Гц, dt = 250 мс)2. **Detrend** (вычитание среднего) → убирает пик на 0 Гц3. **Окно Хана** → устраняет спектральную утечку4. **FFT** → спектральная плотность мощности (PSD, мс²/Гц)5. **Интегрирование** по диапазонам VLF / LF / HF методом трапеций| Диапазон | Частота, Гц | Физиол. смысл ||---|---|---|| HF | 0.15–0.40 | Дыхательные волны. Парасимпатика (вагус). || LF | 0.04–0.15 | Волны Майера. Барорефлекс. Вазомоторный центр. || VLF | 0.015–0.04 | Гормональная регуляция, терморегуляция, кора. |

In [ ]:
fs = 4.0          # Гцdt = 1.0 / fs     # 250 мсt_irr  = np.cumsum(rr) / 1000.0t_uni  = np.arange(t_irr[0], t_irr[-1], dt)cs     = CubicSpline(t_irr, rr)rr_uni = cs(t_uni)N      = len(rr_uni)print(f"Исходных точек : {len(rr)}")print(f"После интерпол.: {N}  (шаг {dt*1000:.0f} мс, {fs:.0f} Гц)")print(f"Длительность   : {t_irr[-1]:.1f} с  ({t_irr[-1]/60:.1f} мин)")print(f"Разрешение     : {fs/N:.4f} Гц")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))ax = axes[0]ax.plot(t_irr, rr, color="steelblue", linewidth=0.8, marker=".", markersize=2)ax.set_ylabel("RR, мс"); ax.set_title("Кардиоритмограмма"); ax.grid(alpha=0.3)ax = axes[1]t0 = t_irr[0]ax.plot(t_irr, rr, "o", color="steelblue", markersize=5, alpha=0.8, label="Исходные удары")ax.plot(t_uni, rr_uni, "-", color="tomato", linewidth=1.2, label="Кубический сплайн (250 мс)")ax.set_xlim(t0, t0 + 30); ax.set_xlabel("Время, с"); ax.set_ylabel("RR, мс")ax.set_title("Первые 30 с: исходные точки и интерполяция")ax.legend(); ax.grid(alpha=0.3)plt.tight_layout(); fig.savefig(FIGURES_DIR / "fig4_cardiorhythmogram.png"); plt.show()

In [ ]:
window  = np.hanning(N)rr_proc = (rr_uni - rr_uni.mean()) * windowX     = np.fft.rfft(rr_proc)freqs = np.fft.rfftfreq(N, d=dt)psd   = np.abs(X)**2 / (fs * np.sum(window**2))psd[1:-1] *= 2def band_power(flo, fhi):    mask = (freqs >= flo) & (freqs <= fhi)    return float(np.trapezoid(psd[mask], freqs[mask]))def band_peak(flo, fhi):    mask = (freqs >= flo) & (freqs <= fhi)    return float(freqs[mask][np.argmax(psd[mask])])VLF, LF, HF = band_power(0.015, 0.040), band_power(0.040, 0.150), band_power(0.150, 0.400)TP           = VLF + LF + HFvlf_f, lf_f, hf_f = band_peak(0.015,0.040), band_peak(0.040,0.150), band_peak(0.150,0.400)IC   = (VLF + LF) / HFLFHF = LF / HFISCA = LF / VLFprint(f"TP   = {TP:.1f} мс²")print(f"VLF  = {VLF:.1f} мс²  ({VLF/TP*100:.1f}%)   [норма: 25–44%]   пик: {vlf_f:.4f} Гц → {1/vlf_f:.0f} с")print(f"LF   = {LF:.1f} мс²  ({LF/TP*100:.1f}%)   [норма: 20–39%]   пик: {lf_f:.4f} Гц → {1/lf_f:.0f} с")print(f"HF   = {HF:.1f} мс²  ({HF/TP*100:.1f}%)   [норма: 15–25%]   пик: {hf_f:.4f} Гц → {1/hf_f:.1f} с")print()print(f"IC      = {IC:.2f}   [норма: 1–5]")print(f"LF/HF   = {LFHF:.2f}   [норма: 0.5–2]")print(f"ISCA    = {ISCA:.2f}   (LF/VLF)")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))bands = [    (0.015, 0.040, "orange",    f"VLF {VLF/TP*100:.1f}%"),    (0.040, 0.150, "limegreen", f"LF  {LF/TP*100:.1f}%"),    (0.150, 0.400, "tomato",    f"HF  {HF/TP*100:.1f}%"),]for flo, fhi, color, label in bands:    mask = (freqs >= flo) & (freqs <= fhi)    ax.fill_between(freqs[mask], psd[mask], alpha=0.45, color=color, label=label)ax.plot(freqs, psd, color="navy", linewidth=0.9)for f in [0.040, 0.150, 0.400]:    ax.axvline(f, color="gray", linewidth=0.8, linestyle=":")ax.set_xlim(0, 0.45)ax.set_xlabel("Частота, Гц"); ax.set_ylabel("Мощность, мс²/Гц")ax.set_title("Спектральная плотность мощности ВСР", fontsize=14, fontweight="bold")ax.legend(fontsize=11); ax.grid(alpha=0.3)plt.tight_layout(); fig.savefig(FIGURES_DIR / "fig5_spectrum.png"); plt.show()

---## 4. Комплексная оценка ПАРС (Баевский)ПАРС = 1 + |А| + |Б| + |В| + |Д1| + |Д2|  (1–10 баллов)| Компонент | Показатель | Смысл ||---|---|---|| А | ЧСС | Тахикардия (+) / Брадикардия (−) || Б | MxDMn, CV | Ригидный ритм (+) / Аритмия (−) || В | MxDMn + АМо | Симпатикотония (+) / Парасимпатикотония (−) || Г | CV, SI | Флаг переходного процесса || Д1 | LF% | Активность вазомоторного центра || Д2 | VLF% | Активность симпатического подкоркового центра |Интерпретация по баллам:- **1–2** 🟢 Оптимальное (рабочее) напряжение- **3–4** 🟡 Умеренное напряжение- **5–6** 🟡 Выраженное напряжение- **7** 🔴 Перенапряжение- **8** 🔴 Истощение- **9–10** 🔴 Срыв адаптации

In [ ]:
LF_pct  = LF  / TP * 100VLF_pct = VLF / TP * 100# А: ЧССA = +2 if HR>90 else (+1 if HR>=80 else (0 if HR>=60 else (-1 if HR>=51 else -2)))# Б: MxDMn / CVB_d = -2 if MxDMn>500 else (-1 if MxDMn>300 else (0 if MxDMn>=150 else (+1 if MxDMn>60 else +2)))B_c = +2 if CV<=2.0 else (+1 if CV<=4.0 else 0)B = B_d if abs(B_d) >= abs(B_c) else B_c# В: MxDMn + АМоC_d = -2 if MxDMn>500 else (-1 if MxDMn>300 else (0 if MxDMn>=150 else (+1 if MxDMn>60 else +2)))C_a = +2 if AMo>80 else (+1 if AMo>=51 else (0 if AMo>=30 else (-1 if AMo>=20 else -2)))C = C_d if abs(C_d) >= abs(C_a) else C_a# Г: флаг нестабильностиG = (CV > 15) and (SI < 15)# Д1: LF%D1 = +2 if LF_pct>=55 else (+1 if LF_pct>=40 else (0 if LF_pct>=20 else (-1 if LF_pct>=11 else -2)))# Д2: VLF%D2 = +2 if VLF_pct>=60 else (+1 if VLF_pct>=45 else (0 if VLF_pct>=25 else (-1 if VLF_pct>=16 else -2)))# Итогpars_val = min(1 + abs(A) + abs(B) + abs(C) + abs(D1) + abs(D2) + (1 if G else 0), 10)state_main = (    "Оптимальное напряжение 🟢" if pars_val<=2 else    "Умеренное напряжение 🟡"   if pars_val<=4 else    "Выраженное напряжение 🟡"  if pars_val<=6 else    "Перенапряжение 🔴"         if pars_val<=7 else    "Истощение 🔴"              if pars_val<=8 else "Срыв адаптации 🔴")state_bb = (    "Норма / удовлетворительная адаптация" if pars_val<=3 else    "Функциональное напряжение"            if pars_val<=5 else    "Перенапряжение / неудовл. адаптация"  if pars_val<=7 else "Истощение / срыв")print(f"А  = {A:+d}   ЧСС = {HR:.1f} уд/мин")print(f"Б  = {B:+d}   MxDMn = {MxDMn:.0f} мс,  CV = {CV:.1f}%")print(f"В  = {C:+d}   MxDMn = {MxDMn:.0f} мс,  АМо = {AMo:.1f}%")print(f"Г  = {'да' if G else 'нет'}")print(f"Д1 = {D1:+d}   LF%  = {LF_pct:.1f}%")print(f"Д2 = {D2:+d}   VLF% = {VLF_pct:.1f}%")print(f"{'─'*55}")flag = ' + 1(Г)' if G else ''print(f"ПАРС = 1 + |{A}|+|{B}|+|{C}|+|{D1}|+|{D2}|{flag} = {pars_val}")print(f"{'─'*55}")print(f"Состояние: {state_main}")print(f"Баевский : {state_bb}")

---## Сводная таблица результатов| Раздел | Главные выводы ||---|---|| Статистика | SDNN, RMSSD, pNN50 в норме → суммарная регуляция сохранена. || Геометрия  | ИН (SI) в норме → нет выраженного централизованного управления. || АКФ        | CC1 показывает уровень инерционности ритма. || Скаттер.   | Форма облака — норма (эллипс вдоль биссектрисы). || Спектр     | Соотношение HF/LF/VLF, индексы IC и LF/HF — баланс ВНС. || ПАРС       | Итоговая комплексная оценка функционального состояния. |> ⚠️ Если запись короче 5 минут, оценка VLF может быть ненадёжной.